**Downloading data**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('santander-customer-transaction-prediction')

print("Path to competition files:", path)

In [1]:
from pathlib import Path
import pandas as pd

def load_housing_data(x):
    # if not tarball_path.is_file():
        # Path("/content/datasets").mkdir(parents=True, exist_ok=True)
        # url = "https://github.com/ageron/data/raw/main/housing.tgz"
        # urllib.request.urlretrieve(url, tarball_path)
    # with tarfile.open(tarball_path) as housing_tarball:
    #     housing_tarball.extractall(path="datasets")
    return pd.read_csv(x, low_memory=False)

train= load_housing_data("/kaggle/input/competitions/santander-customer-transaction-prediction/train.csv")
test= load_housing_data("/kaggle/input/competitions/santander-customer-transaction-prediction/test.csv")

In [11]:
train.head()

,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199
0,train_0,0,8.9255,-6.7863,11.9081,5.0930,11.4607,-9.2834,5.1187,18.6266,...,4.4354,3.9642,3.1364,1.6910,18.5227,-2.3978,7.8784,8.5635,12.7803,-1.0914
1,train_1,0,11.5006,-4.1473,13.8588,5.3890,12.3622,7.0433,5.6208,16.5338,...,7.6421,7.7214,2.5837,10.9516,15.4305,2.0339,8.1267,8.7889,18.3560,1.9518
2,train_2,0,8.6093,-2.7457,12.0805,7.8928,10.5825,-9.0837,6.9427,14.6155,...,2.9057,9.7905,1.6704,1.6858,21.6042,3.1417,-6.5213,8.2675,14.7222,0.3965
3,train_3,0,11.0604,-2.1518,8.9522,7.1957,12.5846,-1.8361,5.8428,14.9250,...,4.4666,4.7433,0.7178,1.4214,23.0347,-1.2706,-2.9275,10.2922,17.9697,-8.9996
4,train_4,0,9.8369,-1.4834,12.8746,6.6375,12.2772,2.4486,5.9405,19.2514,...,-1.4905,9.5214,-0.1508,9.1942,13.2876,-1.5121,3.9267,9.5031,17.9974,-8.8104


In [5]:
corr_matrix = train.corr(numeric_only=True)
corr_matrix["target"].sort_values(ascending=False)

target     1.000000
var_6      0.066731
var_110    0.064275
var_53     0.063399
var_26     0.062422
             ...   
var_76    -0.061917
var_146   -0.063644
var_12    -0.069489
var_139   -0.074080
var_81    -0.080917
Name: target, Length: 201, dtype: float64

In [6]:
train.isna().sum()          # or tranx.isnull().sum() — identical

ID_code    0
target     0
var_0      0
var_1      0
var_2      0
          ..
var_195    0
var_196    0
var_197    0
var_198    0
var_199    0
Length: 202, dtype: int64

In [9]:
train[train.isna().any(axis=1)]

,ID_code,target,var_0,var_1,var_2,var_3,var_4,var_5,var_6,var_7,...,var_190,var_191,var_192,var_193,var_194,var_195,var_196,var_197,var_198,var_199


In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
from sklearn.dummy import DummyClassifier

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# ------------------------------------------------------------------
# 1. Target must be binary 0/1
# ------------------------------------------------------------------
y = train["target"].astype(int)
assert set(np.unique(y)) <= {0, 1}, "target is not binary — AUC not applicable"
print(f"target: n={len(y)} positives={y.sum()} ({y.mean()*100:.2f}%)")

drop_cols    = [c for c in ["ID_code","target"] if c in train.columns]
feature_cols = [c for c in train.columns if c not in drop_cols]
X = train[feature_cols].copy()

num_cols = X.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
print(f"num_cols={len(num_cols)} cat_cols={len(cat_cols)}")

# ------------------------------------------------------------------
# 2. Shared preprocessing (LGBM + XGB need numeric input)
# ------------------------------------------------------------------
pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OrdinalEncoder(handle_unknown="use_encoded_value",
                                  unknown_value=-1)),
    ]), cat_cols),
])

# ------------------------------------------------------------------
# 3. Classifiers
# ------------------------------------------------------------------
lgbm_clf = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=40,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    device_type="gpu",
)

xgb_clf = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    tree_method="hist",          # "gpu_hist" if GPU
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

cat_clf = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    task_type="GPU",
)

models = {
    "lightgbm": Pipeline([("pre", pre), ("clf", lgbm_clf)]),
    "xgboost":  Pipeline([("pre", pre), ("clf", xgb_clf)]),
    "catboost": cat_clf,   # gets raw frame + native cat features
}

# ------------------------------------------------------------------
# 4. 10-fold CV with AUC
# ------------------------------------------------------------------
cat_idx = [feature_cols.index(c) for c in cat_cols]
kf = KFold(n_splits=5, shuffle=True, random_state=42)
folds = list(kf.split(X))

results = {n: {"train": [], "val": [], "oof": np.zeros(len(y))} for n in models}
base_scores = []

for fold, (tr_idx, val_idx) in enumerate(folds, start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    base = DummyClassifier(strategy="prior").fit(X_tr, y_tr)
    bs = roc_auc_score(y_val, base.predict_proba(X_val)[:, 1])
    base_scores.append(bs)

    line = f"Fold {fold:2d} | baseline={bs:.4f}"
    for name, est in models.items():
        m = clone(est)
        if name == "catboost":
            m.fit(X_tr, y_tr, cat_features=cat_idx)
        else:
            m.fit(X_tr, y_tr)

        tr_auc = roc_auc_score(y_tr, m.predict_proba(X_tr)[:, 1])
        vl_auc = roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])
        results[name]["train"].append(tr_auc)
        results[name]["val"].append(vl_auc)
        results[name]["oof"][val_idx] = m.predict_proba(X_val)[:, 1]
        line += f" | {name}={vl_auc:.4f}"
    print(line)

# ------------------------------------------------------------------
# 5. Summary (higher AUC = better)
# ------------------------------------------------------------------
def pooled_auc(name):
    return roc_auc_score(y, results[name]["oof"])

summary = pd.DataFrame([{
    "model": n,
    "mean_train_auc": np.mean(results[n]["train"]),
    "mean_val_auc":   np.mean(results[n]["val"]),
    "std_val_auc":    np.std(results[n]["val"]),
    "pooled_oof_auc": pooled_auc(n),
    "train_val_gap":  np.mean(results[n]["train"]) - np.mean(results[n]["val"]),
} for n in models]).sort_values("pooled_oof_auc", ascending=False)

print("\n=== AUC comparison (higher is better) ===")
print(summary.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
print(f"\nBaseline (random) AUC: {np.mean(base_scores):.5f}")

# ------------------------------------------------------------------
# 6. Blends
# ------------------------------------------------------------------
blend_all = np.mean([results[n]["oof"] for n in models], axis=0)
print(f"Equal-weight blend (all 3) AUC: {roc_auc_score(y, blend_all):.5f}")

blend_2 = np.mean([results["xgboost"]["oof"],
                   results["lightgbm"]["oof"]], axis=0)
print(f"XGB+LGBM blend AUC:            {roc_auc_score(y, blend_2):.5f}")

target: n=200000 positives=20098 (10.05%)
num_cols=200 cat_cols=0


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature 

Fold  1 | baseline=0.5000 | lightgbm=0.8947 | xgboost=0.8943 | catboost=0.8946


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Default metric period is 5 because AUC is/are not implemented for GPU


Fold  2 | baseline=0.5000 | lightgbm=0.8983 | xgboost=0.8962 | catboost=0.8968


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Default metric period is 5 because AUC is/are not implemented for GPU


Fold  3 | baseline=0.5000 | lightgbm=0.8907 | xgboost=0.8894 | catboost=0.8900


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Default metric period is 5 because AUC is/are not implemented for GPU


Fold  4 | baseline=0.5000 | lightgbm=0.8942 | xgboost=0.8930 | catboost=0.8942


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Default metric period is 5 because AUC is/are not implemented for GPU


Fold  5 | baseline=0.5000 | lightgbm=0.8985 | xgboost=0.8978 | catboost=0.8981

=== AUC comparison (higher is better) ===
   model  mean_train_auc  mean_val_auc  std_val_auc  pooled_oof_auc  train_val_gap
lightgbm         0.99130       0.89526      0.00288         0.89524        0.09605
catboost         0.93015       0.89473      0.00274         0.89472        0.03542
 xgboost         0.98493       0.89416      0.00288         0.89415        0.09078

Baseline (random) AUC: 0.50000
Equal-weight blend (all 3) AUC: 0.89705
XGB+LGBM blend AUC:            0.89611
